In [2]:
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

In [3]:
# STAGE 1: LOAD RAW DATA (2009-2010 ONLY)
# =====================================================================
print("[1/6] Loading raw data for the year 2009-2010...")
# Load the single dataset directly
# Use the repository Excel file located at data/raw/raw_data.xlsx
file_path = '../data/raw/raw_data.xlsx'

# Read the Excel file (xls/xlsx)
df = pd.read_excel(file_path)
print(f"      -> Initial total rows: {len(df):,}")
# Capture the initial row count for final statistics
initial_row_count = len(df)
print(f"      -> Initial total rows: {initial_row_count:,}")

[1/6] Loading raw data for the year 2009-2010...
      -> Initial total rows: 525,461
      -> Initial total rows: 525,461


In [4]:
# STAGE 2: BASIC CLEANING
# =====================================================================
print("\n[2/6] Standardizing data types and removing duplicates...")
# Convert InvoiceDate to datetime object for the system to process correctly
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce').dt.date

# Drop exact duplicate rows
df = df.drop_duplicates()
print(f"      -> Rows remaining after dropping duplicates: {len(df):,}")


[2/6] Standardizing data types and removing duplicates...
      -> Rows remaining after dropping duplicates: 518,595


In [5]:
# STAGE 3: REQUIREMENT 1 - HANDLE ALPHABETICAL STOCK CODES
# =====================================================================
print("\n[3/6] Analyzing and filtering noisy StockCodes...")
df['StockCode'] = df['StockCode'].astype(str)

# Find StockCodes that contain only letters (isalpha = True)
alpha_stocks = df[df['StockCode'].str.isalpha()]['StockCode'].unique()

print("      -> List of purely alphabetical StockCodes (unique):")
print(f"         {list(alpha_stocks)}")
print(f"      -> Total identified: {len(alpha_stocks)} noisy codes.")

# FILTER OUT: These codes are typically postage fees, discounts, or manual adjustments, 
# not actual physical SKUs. Removing them prevents the AI from learning incorrect patterns.
df = df[~df['StockCode'].isin(alpha_stocks)]
print(f"      -> Rows remaining after removing noisy StockCodes: {len(df):,}")

# =====================================================================
# STAGE 4: REQUIREMENT 2 - FILTER QUANTITY AND PRICE <= 0
# =====================================================================
print("\n[4/6] Removing invalid orders (Quantity or Price <= 0)...")
df = df[(df['Quantity'] > 0) & (df['Price'] > 0)]
print(f"      -> Rows remaining after filtering: {len(df):,}")

# =====================================================================
# STAGE 5: REQUIREMENT 3 - DROP CUSTOMER ID AND COUNTRY COLUMNS
# =====================================================================
print("\n[5/6] Dropping unused columns for the machine learning models...")
# This list comprehension dynamically catches the column name whether it has a space or not
cols_to_drop = [col for col in ['Customer ID', 'CustomerID', 'Country'] if col in df.columns]

df = df.drop(columns=cols_to_drop)
print(f"      -> Successfully dropped columns: {cols_to_drop}")


[3/6] Analyzing and filtering noisy StockCodes...
      -> List of purely alphabetical StockCodes (unique):
         ['POST', 'D', 'DOT', 'M', 'PADS', 'ADJUST', 'DCGSSGIRL', 'GIFT', 'DCGSLBOY', 'm', 'DCGSSBOY', 'DCGSLGIRL', 'S', 'B', 'AMAZONFEE']
      -> Total identified: 15 noisy codes.
      -> Rows remaining after removing noisy StockCodes: 515,884

[4/6] Removing invalid orders (Quantity or Price <= 0)...
      -> Rows remaining after filtering: 502,608

[5/6] Dropping unused columns for the machine learning models...
      -> Successfully dropped columns: ['Customer ID', 'Country']


In [6]:
def assign_category(text):
    if pd.isna(text):
        return 'Others'
    
    text = text.upper()
    
    # Remove general 'GLASS' keyword from kitchen group
    kitchen_keywords = ['MUG', 'CUP', 'BOWL', 'PLATE', 'CAKESTAND', 'TIN', 
                        'CAKE', 'BAKING', 'PAN', 'FORK', 'SPOON', 'KNIFE', 
                        'CUTLERY', 'APRON', 'JAR']
                        
    lighting_keywords = ['LIGHT', 'LAMP', 'LED', 'LANTERN', 'BULB', 'CANDLE']
    bag_keywords = ['BAG', 'SHOPPER', 'TOTE']
    party_keywords = ['HEART', 'ORNAMENT', 'BUNTING', 'FRAME', 'VINTAGE', 'RETRO', 'CHRISTMAS', 'PARTY']
    stationery_keywords = ['WRAP', 'PAPER', 'TAPE', 'PEN', 'RIBBON', 'CARD', 'PENCIL']
    
    # --- CHECKING PRIORITY FROM TOP TO BOTTOM ---
    
    # Priority 1: Catch Lights first (GLASS LIGHT will be caught here)
    if any(word in text for word in lighting_keywords):
        return 'Lighting & Candles'
        
    # Priority 2: Catch Bags
    elif any(word in text for word in bag_keywords):
        return 'Bags'
        
    # Priority 3: Catch Kitchenware (Add explicit terms for glass)
    elif any(word in text for word in kitchen_keywords) or 'WINE GLASS' in text or 'WATER GLASS' in text:
        return 'Tableware & Kitchen'
        
    # Priority 4: Party Decorations
    elif any(word in text for word in party_keywords):
        return 'Party & Decoration'
        
    # Priority 5: Stationery
    elif any(word in text for word in stationery_keywords):
        return 'Stationery & Crafts'
        
    # Final fallback
    else:
        return 'Others'

# Resolve the source column defensively because the notebook may be rerun from an already-categorized state.
description_col = next((col for col in ['description', 'Description', 'DESCRIPTION'] if col in df.columns), None)
if description_col is not None:
    category_values = df[description_col].apply(assign_category)
    removed_columns = [description_col, 'Category', 'category']
else:
    category_col = next((col for col in ['Category', 'category'] if col in df.columns), None)
    if category_col is None:
        raise KeyError(f"Could not find a description or category column. Available columns: {list(df.columns)}")
    category_values = df[category_col]
    removed_columns = ['Category', 'category']

# Rebuild the dataframe schema so the CSV contains exactly one Category column right after StockCode.
base_columns = [col for col in df.columns if col not in removed_columns]
if 'StockCode' in base_columns:
    category_position = base_columns.index('StockCode') + 1
else:
    category_position = 0
df = df[base_columns].copy()
df.insert(category_position, 'Category', category_values.to_numpy())

print(df[['Category']].head(100))
print(df['Category'].value_counts())


               Category
0    Lighting & Candles
1    Lighting & Candles
2    Lighting & Candles
3    Party & Decoration
4                Others
..                  ...
96               Others
97               Others
98   Party & Decoration
99               Others
100              Others

[100 rows x 1 columns]
Category
Others                 205813
Tableware & Kitchen     93186
Party & Decoration      82752
Lighting & Candles      44947
Bags                    43792
Stationery & Crafts     32118
Name: count, dtype: int64


In [8]:
# =====================================================================
# STAGE 6: CATEGORIZATION, TRAIN/TEST SPLIT & EXPORT
# =====================================================================
print("\n[6/6] Standardizing, Splitting data (Train <= Oct 2010) and exporting...")

# 1. Rename columns and assign Categories
rename_map = {
    'Invoice': 'invoice_no', 'InvoiceNo': 'invoice_no', 'inovoice_no': 'invoice_no',
    'Stock_code': 'stock_code', 'StockCode': 'stock_code',
    'InovoiceDate': 'order_date', 'InvoiceDate': 'order_date',
    'Description': 'description', 'Quantity': 'quantity', 'Price': 'price'
}
df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

def assign_category(text):
    if pd.isna(text): return 'Others'
    text = str(text).upper()
    kitchen_keywords = ['MUG', 'CUP', 'BOWL', 'PLATE', 'CAKESTAND', 'TIN', 'CAKE', 'BAKING', 'PAN', 'FORK', 'SPOON', 'KNIFE', 'CUTLERY', 'APRON', 'JAR']
    lighting_keywords = ['LIGHT', 'LAMP', 'LED', 'LANTERN', 'BULB', 'CANDLE']
    bag_keywords = ['BAG', 'SHOPPER', 'TOTE']
    party_keywords = ['HEART', 'ORNAMENT', 'BUNTING', 'FRAME', 'VINTAGE', 'RETRO', 'CHRISTMAS', 'PARTY']
    stationery_keywords = ['WRAP', 'PAPER', 'TAPE', 'PEN', 'RIBBON', 'CARD', 'PENCIL']

    if any(word in text for word in lighting_keywords): return 'Lighting & Candles'
    elif any(word in text for word in bag_keywords): return 'Bags'
    elif any(word in text for word in kitchen_keywords) or 'WINE GLASS' in text or 'WATER GLASS' in text: return 'Tableware & Kitchen'
    elif any(word in text for word in party_keywords): return 'Party & Decoration'
    elif any(word in text for word in stationery_keywords): return 'Stationery & Crafts'
    else: return 'Others'

if 'description' in df.columns:
    df['category'] = df['description'].apply(assign_category)
    df = df.drop(columns=['description'])
else:
    df['category'] = 'Others'

# 2. TRAIN/TEST SPLIT (PREVENT DATA LEAKAGE)
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')

# Extract Train set (Up to Oct 31, 2010)
df_train = df[df['order_date'] <= '2010-10-31']
# Extract Test set (From Nov 01, 2010 onwards)
df_test = df[df['order_date'] >= '2010-11-01']

# 3. Reorder columns for consistency
desired_order = ['order_date', 'invoice_no', 'stock_code', 'category', 'quantity', 'price']
df_train = df_train[[col for col in desired_order if col in df_train.columns]]
df_test = df_test[[col for col in desired_order if col in df_test.columns]]
df = df[[col for col in desired_order if col in df.columns]]

# 4. Export files to target directory
output_dir = '../data/processed'
os.makedirs(output_dir, exist_ok=True)
df_train.to_csv(f'{output_dir}/train_invoices.csv', index=False)
df_test.to_csv(f'{output_dir}/test_invoices.csv', index=False)
df.to_csv(f'{output_dir}/data_clean.csv', index=False) # Consolidated backup file

print(f" 📦 TRAIN SET (<= Oct 31) for SQL & K-Means: {len(df_train):,} rows")
print(f" 📦 TEST SET  (>= Nov 01) for Simulation:    {len(df_test):,} rows")


[6/6] Standardizing, Splitting data (Train <= Oct 2010) and exporting...
 📦 TRAIN SET (<= 31/10) for SQL & K-Means: 406,195 rows
 📦 TEST SET  (>= 01/11) for Simulation:    96,413 rows
